# Regresión logística: clasificación de vinos

En este notebook vamos a aprender cómo funciona **la regresión logística**, uno de los modelos más utilizados para **problemas de clasificación** en machine learning.

Para hacerlo más intuitivo, trabajaremos con el **Wine Dataset**, un conjunto de datos que contiene resultados de análisis químicos de vinos producidos en la misma región de Italia pero provenientes de **tres tipos diferentes de uva**. Cada vino está descrito por **13 características químicas**, como el nivel de alcohol, flavonoides, cenizas, etc.

A lo largo del notebook veremos cómo:

1. **Explorar y preparar los datos**
2. **Entrenar un modelo de regresión logística**
3. **Evaluar su rendimiento**
4. **Visualizar la frontera de decisión**
5. **Extender el modelo de clasificación binaria a clasificación multiclase**

Para facilitar la comprensión, en algunos pasos utilizaremos **solo dos variables**, lo que nos permitirá visualizar cómo el modelo separa las clases en un plano.

El objetivo es entender **la intuición detrás del modelo**, más que usar directamente una librería de alto nivel.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score

### Cargando el dataset de vinos

Vamos a trabajar con un dataset clásico en machine learning: **Wine Dataset**.

Contiene resultados de análisis químicos de vinos producidos en la misma región de Italia pero procedentes de **tres tipos diferentes de uva (cultivars)**. Para cada vino se han medido **13 propiedades químicas**, como por ejemplo:

- alcohol
- alcalinidad de la ceniza (ash)
- flavonoides
- magnesio
- etc.

Nuestro objetivo será **clasificar los vinos según su tipo** utilizando estas características.

Primero cargamos el dataset desde un repositorio público y mostramos las primeras filas para entender su estructura.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/Jorgeprevi/ML_bootcamp_2026/refs/heads/main/regresion_logistica/vinos.csv', sep = ",")
data.head()

### Preparando un problema de clasificación binaria

La **regresión logística clásica** es un modelo pensado para **clasificación binaria** (dos clases).

Nuestro dataset tiene **tres tipos de vino**, pero para empezar vamos a simplificar el problema:

- Nos quedamos solo con **dos clases** (1 y 2)
- Usamos únicamente **dos variables**:
  - `alcohol`
  - `ash`

Esto nos permitirá **visualizar el problema en 2D**, lo cual facilita mucho entender cómo funciona el modelo.

Además, convertimos las etiquetas en formato **binario (0 y 1)** utilizando `label_binarize`.

Esto es necesario porque el modelo de regresión logística trabaja con valores binarios.

También incluimos un pequeño ejemplo para ver cómo funciona el proceso de **binarización de etiquetas**.

In [ ]:
reduced = data[data['class'] <= 2]
X = reduced[['alcohol', 'ash']].values
y = label_binarize(reduced['class'].to_numpy(), classes=[1, 2])[:, 0]

example = np.copy(data['class'].to_numpy())
np.random.shuffle(example)
example = example[:10]

print('original:', example)

example = label_binarize(example, classes=list(set(example)))

print('binarized:', example)
print('1s vs all:', example[:, 0])

### Dividiendo el dataset en entrenamiento y test

Para evaluar correctamente un modelo necesitamos separar los datos en dos grupos:

- **Train set** → datos que el modelo usa para aprender
- **Test set** → datos nuevos para evaluar el rendimiento

En este caso usaremos:

- **75% de los datos para entrenamiento**
- **25% para test**

Esto nos permitirá comprobar si el modelo **generaliza bien a datos que no ha visto antes**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)
print('train:', len(X_train), 'test:', len(X_test))

### Visualizando los datos

Antes de entrenar un modelo es muy útil **visualizar los datos**.

Como estamos usando solo dos variables (`alcohol` y `ash`), podemos representar cada vino como un punto en un plano:

- eje X → alcohol
- eje Y → ash

Cada clase se mostrará con **un color y marcador diferente**.

Esto nos permitirá ver si las clases parecen **separables**, algo importante para que la regresión logística funcione bien.

In [ ]:
MARKERS = ['+', 'x', '.']
COLORS = ['red', 'green', 'blue']

def plot_points(xy, labels):
    
    for i, label in enumerate(set(labels)):
        points = np.array([xy[j,:] for j in range(len(xy)) if labels[j] == label])
        marker = MARKERS[i % len(MARKERS)]
        color = COLORS[i % len(COLORS)]
        plt.scatter(points[:,0], points[:,1], marker=marker, color=color)

plot_points(X_train, y_train)

## Implementando la regresión logística

Ahora vamos a implementar los componentes básicos del modelo.

**Función sigmoide**

La regresión logística utiliza la **función sigmoide** para transformar cualquier valor en una **probabilidad entre 0 y 1**:

\[
\sigma(x) = \frac{1}{1 + e^{-x}}
\]

Esto permite interpretar la salida del modelo como:

> Probabilidad de pertenecer a la clase positiva.


**Función de coste**

La función de coste mide **qué tan bien o mal está funcionando el modelo**.

En regresión logística usamos la **log-loss (entropía cruzada)**, que penaliza fuertemente las predicciones incorrectas.



**Gradiente**

El gradiente indica **cómo debemos modificar los parámetros del modelo** para reducir el error.

El algoritmo de optimización utilizará este gradiente para encontrar los mejores parámetros.

In [ ]:
def sigmoid(X):
    return 1 / (1 + np.exp(-X))

def cost(theta, X, y):
    theta = theta[:,None]
    y = y[:,None]
    
    hyp = sigmoid(X.dot(theta))
    pos = np.multiply(-y, np.log(hyp))
    neg = np.multiply((1 - y), np.log(1 - hyp))
    
    return np.sum(pos - neg) / (len(X))

def gradient(theta, X, y):
    theta = theta[:,None]
    y = y[:,None]
    
    error = sigmoid(X.dot(theta)) - y
    return X.T.dot(error) / len(X)

### Entrenando el modelo

Ahora entrenamos el modelo de regresión logística.

Primero añadimos una columna de **unos (bias)** a las variables de entrada. Esto permite al modelo aprender un **intercept**.

Después usamos `fmin_tnc` de `scipy.optimize`, que es un algoritmo de optimización numérica que busca los parámetros `theta` que **minimizan la función de coste**.

El resultado es el vector de parámetros del modelo.

In [ ]:
from scipy.optimize import fmin_tnc  

def train(X, y):
    X = np.insert(X, 0, np.ones(len(X)), axis=1)
    theta = np.zeros(X.shape[1])
    result = fmin_tnc(func=cost, x0=theta, fprime=gradient, args=(X, y))
    
    return result[0]

theta = train(X_train, y_train)
print('theta: ', theta)

### Haciendo predicciones y evaluando el modelo

Una vez entrenado el modelo, podemos usarlo para **predecir la clase de nuevos datos**.

El proceso es:

1. Calculamos la probabilidad usando la función sigmoide.
2. Si la probabilidad es **mayor o igual a 0.5**, clasificamos como clase 1.
3. Si es menor, clasificamos como clase 0.

Después evaluamos el modelo usando tres métricas comunes:

- **Accuracy** → proporción de predicciones correctas
- **Precision** → calidad de las predicciones positivas
- **Recall** → capacidad de detectar correctamente los positivos

Estas métricas nos ayudan a entender **cómo de bien está funcionando el modelo**.

In [ ]:
def predict(X, theta):
    X = np.insert(X, 0, np.ones(len(X)), axis=1)
    theta = np.asarray(theta)          # asegurar ndarray
    return (sigmoid(X @ theta) >= 0.5).astype(int)

predictions = predict(X_test, theta)

print('accuracy:', accuracy_score(y_test, predictions))
print('precision:', precision_score(y_test, predictions, average='macro'))
print('recall:', recall_score(y_test, predictions, average='macro'))

### Visualizando la frontera de decisión

Una de las ventajas de usar solo dos variables es que podemos **visualizar la frontera de decisión del modelo**.

La frontera de decisión es la línea que separa las regiones donde el modelo predice:

- clase 0
- clase 1

Para calcularla:

1. Generamos una rejilla de puntos en el plano
2. Aplicamos el modelo a cada punto
3. Dibujamos la línea donde cambia la predicción

Esto nos permite ver **cómo está separando el modelo las dos clases**.

In [ ]:
from matplotlib import cm

def plot_boundary(X, pred):
    
    x_min, x_max = X[:,0].min() - .1, X[:,0].max() + .1
    y_min, y_max = X[:,1].min() - .1, X[:,1].max() + .1
    
    xs, ys = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    xys = np.column_stack([xs.ravel(), ys.ravel()])
    zs = pred(xys).reshape(xs.shape)

    plt.contour(xs, ys, zs, colors='black')
        
plot_points(X_train, y_train)
plot_boundary(X_train, lambda x: predict(x, theta))

### Clasificación multiclase (One-vs-Rest)

Hasta ahora hemos trabajado con **dos clases**, pero el dataset original tiene **tres tipos de vino**.

Una forma común de extender la regresión logística a múltiples clases es la estrategia **One-vs-Rest (OvR)**:

1. Entrenamos **un modelo por cada clase**
2. Cada modelo aprende a distinguir:
   
   "esta clase vs todas las demás"

En este caso entrenamos **tres modelos**:

- clase 1 vs resto
- clase 2 vs resto
- clase 3 vs resto

Para hacer una predicción:

- calculamos la probabilidad para cada modelo
- elegimos la clase con **mayor probabilidad**

Finalmente evaluamos el modelo y visualizamos la nueva frontera de decisión.

In [ ]:
X = data[['alcohol', 'flavanoids']].to_numpy()
y = data[['class']].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

y_train = label_binarize(y_train, classes=[1, 2, 3])

plot_points(X_train, y_train.argmax(axis=1))

In [ ]:
def predict_multi(X, thetas):
    X = np.insert(X, 0, np.ones(len(X)), axis=1)
    preds = [sigmoid(X @ np.asarray(t)) for t in thetas]
    return np.column_stack(preds).argmax(axis=1)

thetas = [train(X_train, y_train[:,i]) for i in range(0,3)]
predictions = predict_multi(X_test, thetas) + 1

print('accuracy:', accuracy_score(y_test, predictions))
print('precision:', precision_score(y_test, predictions, average='macro'))
print('recall:', recall_score(y_test, predictions, average='macro'))

plot_points(X_train, y_train.argmax(axis=1))
plot_boundary(X_train, lambda x: predict_multi(x, thetas))

## Visualizando las probabilidades de cada clase

Hasta ahora hemos visto la **frontera de decisión**, es decir, la línea donde el modelo cambia de una clase a otra.

Pero la regresión logística realmente calcula **probabilidades**.

En este ejemplo entrenamos tres modelos con el enfoque **One-vs-Rest**, por lo que cada modelo estima:

> Probabilidad de que un punto pertenezca a una clase concreta.

Para entender mejor cómo se comporta el modelo vamos a visualizar:

- un **mapa de calor de probabilidad** para cada clase
- los **puntos reales del dataset**

Las zonas más intensas indican **mayor probabilidad de pertenecer a esa clase**.

In [ ]:
def plot_points(xy, labels, ax=None):
    if ax is None:
        ax = plt.gca()
        
    for i, label in enumerate(set(labels)):
        points = np.array([xy[j, :] for j in range(len(xy)) if labels[j] == label])
        marker = MARKERS[i % len(MARKERS)]
        color = COLORS[i % len(COLORS)]
        ax.scatter(points[:, 0], points[:, 1], marker=marker, color=color)

# y_train está en formato one-hot, usamos argmax para recuperar la clase
y_labels = y_train.argmax(axis=1)

# Definimos los límites del plano donde dibujaremos la visualización
# Añadimos un pequeño margen para que los puntos no queden pegados al borde
x_min, x_max = X_train[:, 0].min() - .2, X_train[:, 0].max() + .2
y_min, y_max = X_train[:, 1].min() - .2, X_train[:, 1].max() + .2

# Creamos una rejilla de puntos (grid) sobre el plano
xs, ys = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)
grid = np.column_stack([xs.ravel(), ys.ravel()])
grid_with_bias = np.insert(grid, 0, np.ones(len(grid)), axis=1)
fig, axes = plt.subplots(1, len(thetas), figsize=(15, 4))

# Iteramos sobre cada modelo (uno por clase en One-vs-Rest)
for i, theta in enumerate(thetas):
    # Calculamos la probabilidad de esa clase para cada punto
    probs = sigmoid(grid_with_bias @ np.asarray(theta)).reshape(xs.shape)
    ax = axes[i]
    c = ax.contourf(xs, ys, probs, levels=30, cmap=cm.RdYlBu)
    plot_points(X_train, y_labels, ax=ax)
    ax.set_title(f"Probabilidad clase {i+1}")
    ax.set_xlabel("alcohol")
    ax.set_ylabel("flavanoids")
    fig.colorbar(c, ax=ax)
plt.tight_layout()
plt.show()